In [1]:
prefix_path = '../..'

import sys
sys.path.append(prefix_path)

import logging
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

from detection_baselines.detection_stats import load_stats_dataset, stats_collate_fn, ThresholdDetector, train_threshold_detector, eval_threshold_detector
from egh_vlm.utils import Qwen3ModelBundle, load_phd_dataset, split_stratified, create_balanced_dataloader

In [2]:
DATASET_PATH = f'{prefix_path}/data/phd/phd_base_qwen3.json'
IMG_FOLDER_PATH = f'{prefix_path}/data/phd/images'
FEATURES_PATH = f'{prefix_path}/data/phd/features/features_stats_phd_base_qwen3.pt'
OUTPUT_PATH = f'{prefix_path}/data/phd/evaluations/evaluation_stats_threshold_phd_base_qwen3.json'
LOGGING_PATH = f'{prefix_path}/data/logs/log_stats_threshold_phd_base_qwen3.txt'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
TRAIN_RATIO = 0.7

Using device: cuda


#### Load Dataset

In [3]:
dataset = load_phd_dataset(
    dataset_path=DATASET_PATH,
    img_folder_path=IMG_FOLDER_PATH,
)
print(f'Length of dataset: {len(dataset)}')

features = load_stats_dataset(FEATURES_PATH)
print(f'Length of features: {len(features)}')

Successfully load the PhD dataset with: 33688 samples.
Length of dataset: 33688
Length of features: 33688


In [4]:
train_dataset, test_dataset = split_stratified(features, train_ratio=TRAIN_RATIO, random_state=42)
train_dataloader = create_balanced_dataloader(
    train_dataset,
    batch_size=32,
    collate_fn=stats_collate_fn,
    seed=42,
)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=32,
    collate_fn=stats_collate_fn,
    shuffle=True,
)

#### Experiment

In [5]:
logging.basicConfig(filename=LOGGING_PATH, level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

results = {}

for metric_type in ['entropy_mean', 'entropy_max', 'prob_mean', 'prob_max']:
    detector = ThresholdDetector(metric_type=metric_type)
    train_result = train_threshold_detector(detector, train_dataloader)
    eval_result = eval_threshold_detector(detector, test_dataloader)

    logging.info(f'[{metric_type}] threshold={train_result["threshold"]:.4f}, train_f1={train_result["f1"]:.4f}')
    logging.info(f'[{metric_type}] acc={eval_result["acc"]:.4f}, f1={eval_result["f1"]:.4f}, pr_auc={eval_result["pr_auc"]:.4f}')

    results[metric_type] = {
        'threshold': train_result['threshold'],
        'acc': eval_result['acc'],
        'f1': eval_result['f1'],
        'pr_auc': eval_result['pr_auc'],
    }
    print(f'[{metric_type}] threshold={train_result["threshold"]:.4f} | acc={eval_result["acc"]:.4f}, f1={eval_result["f1"]:.4f}, pr_auc={eval_result["pr_auc"]:.4f}')

logger = logging.getLogger()
for handler in logger.handlers[:]:
    handler.close()
    logger.removeHandler(handler)

[entropy_mean] threshold=0.7890 | acc=0.2085, f1=0.3431, pr_auc=0.2305
[entropy_max] threshold=2.8821 | acc=0.2069, f1=0.3428, pr_auc=0.2013
[prob_mean] threshold=0.7624 | acc=0.2102, f1=0.3430, pr_auc=0.2343
[prob_max] threshold=1.0000 | acc=0.2069, f1=0.3428, pr_auc=0.1725


In [6]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
with open(OUTPUT_PATH, 'w') as f:
    json.dump(results, f, indent=2)